# Session 7 · Case study: honest validation on EBTI decisions and churn

*Author: course team · Licence: CC-BY-4.0*

This workbook contains the practice tasks of all three blocks of Session 7:

| Part | Block | Task |
|---|---|---|
| A | 0:00–0:45 | Cross-validate the regression and logistic models of Session 6 and the heading classifier, and report the scores with confidence intervals |
| B | 1:00–1:45 | Tune the ridge penalty and the regularisation of the text classifier; compare random and time-based validation on the EBTI decisions |
| C | 2:00–2:45 | Demonstrate a leaking workflow (word selection outside the CV; the customs' justification as a feature), then fix it and compare the scores |

**Data.** The sample `case-study/data/train_sample.parquet` (50,000 BTI decisions 2017–2023, prepared with `case-study/prepare_data.py`) and the IBM Telco churn data (downloaded from GitHub). The heading classifier is TF-IDF + a linear model (`SGDClassifier`); Session 13 explains how it works, here it is a black box. Running the whole notebook takes a few minutes.

Tasks for you are marked **Task**. Each part ends with a short question to answer in a Markdown cell.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, r2_score, roc_auc_score
from sklearn.model_selection import (GridSearchCV, GroupKFold, KFold, StratifiedKFold, TimeSeriesSplit,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

# find the repository root (the folder that contains case-study/)
ROOT = Path.cwd()
while not (ROOT / "case-study").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "case-study" / "data"

dec = pd.read_parquet(DATA / "train_sample.parquet").sort_values("start_date", ignore_index=True)
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet")
dec = dec.merge(nomenclature[["heading", "section"]], on="heading", how="left")
dec["section"] = dec["section"].fillna("unknown")
print(dec.shape, dec["start_date"].min().date(), dec["start_date"].max().date())

past = (dec["start_date"].dt.year <= 2021).to_numpy()     # development years 2017-2021
print("development:", past.sum(), " out-of-time test (2022-2023):", (~past).sum())

A helper for the bootstrap confidence interval of any metric on a fixed set of predictions (theory page 01).

In [ ]:
def bootstrap_ci(y_true, y_pred, metric, n_boot=1000, seed=0):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        stats.append(metric(y_true[i], y_pred[i]))
    return np.percentile(stats, [2.5, 97.5])

## Part A · Cross-validated scores with confidence intervals

### A1 · Regression of the description length (Session 6)

Target: log(number of characters of the description). Features: year, language (rare languages grouped) and section, as dummy columns. We report

1. the 5-fold cross-validated R² (mean and standard deviation across folds), and
2. the R² on the out-of-time test set (2022–2023) with a bootstrap 95 % interval.

In [ ]:
top = dec["language"].value_counts().index[:8]
reg_frame = pd.DataFrame({"year": dec["start_date"].dt.year - 2017,
                          "language": dec["language"].where(dec["language"].isin(top), "other"),
                          "section": dec["section"]})
X_reg = pd.get_dummies(reg_frame, columns=["language", "section"], drop_first=True, dtype=float)
y_reg = np.log(dec["description"].str.len())

lin = make_pipeline(StandardScaler(), LinearRegression())
cv5 = KFold(5, shuffle=True, random_state=0)
r2 = cross_val_score(lin, X_reg, y_reg, cv=cv5, scoring="r2")
print("R² per fold:", r2.round(3), " mean", r2.mean().round(3), " sd", r2.std().round(3))

lin.fit(X_reg[past], y_reg[past])
pred = lin.predict(X_reg[~past])
print("out-of-time R²:", round(r2_score(y_reg[~past], pred), 3),
      " 95% CI:", bootstrap_ci(y_reg[~past], pred, r2_score).round(3))

Compare the two numbers. Note that the year enters as a number: a model trained on 2017–2021 has to extrapolate the trend to 2022–2023.

### A2 · Logistic regression for Telco churn (Session 6)

We report cross-validated ROC AUC and a bootstrap interval on a stratified 25 % test set.

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
y_churn = (telco["Churn"] == "Yes").astype(int)
X_churn = telco.drop(columns=["customerID", "Churn"])
num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = [c for c in X_churn.columns if c not in num]
churn_model = make_pipeline(
    make_column_transformer((StandardScaler(), num), (OneHotEncoder(handle_unknown="ignore"), cat)),
    LogisticRegression(max_iter=1000))

skf = StratifiedKFold(5, shuffle=True, random_state=0)
auc = cross_val_score(churn_model, X_churn, y_churn, cv=skf, scoring="roc_auc")
print("AUC per fold:", auc.round(3), " mean", auc.mean().round(3), " sd", auc.std().round(3))

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_churn, y_churn, test_size=0.25, stratify=y_churn, random_state=0)
proba = churn_model.fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1]
print("test AUC:", round(roc_auc_score(yc_te, proba), 3),
      " 95% CI:", bootstrap_ci(yc_te, proba, roc_auc_score).round(3))

### A3 · The heading classifier

Accuracy and macro-F1 of the text classifier, with random 5-fold CV and an out-of-time test (2022–2023). Stratified folds are not possible here: many headings have fewer than five decisions in the sample.

In [ ]:
text_model = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True),
                           SGDClassifier(alpha=1e-5, random_state=0, n_jobs=-1))
acc = cross_val_score(text_model, dec["description"], dec["heading"], cv=cv5, scoring="accuracy")
print("accuracy per fold (random 5-fold):", acc.round(3), " mean", acc.mean().round(3), " sd", acc.std().round(3))

text_model.fit(dec["description"][past], dec["heading"][past])
y_true, y_pred = dec["heading"][~past].to_numpy(), text_model.predict(dec["description"][~past])
macro_f1 = lambda a, b: f1_score(a, b, average="macro")
print("out-of-time accuracy:", round(accuracy_score(y_true, y_pred), 3),
      " 95% CI:", bootstrap_ci(y_true, y_pred, accuracy_score, n_boot=300).round(3))
print("out-of-time macro-F1:", round(macro_f1(y_true, y_pred), 3),
      " 95% CI:", bootstrap_ci(y_true, y_pred, macro_f1, n_boot=300).round(3))

**Task A.** Collect the results in one table (model, metric, CV mean ± sd, out-of-time score, 95 % interval). Is the out-of-time score inside the range you would expect from cross-validation? Which estimate is most uncertain, and why is the macro-F1 interval wider than the accuracy interval?

In [ ]:
# your table here, for example:
# pd.DataFrame([{"model": "linear regression", "metric": "R²", "cv_mean": ..., "cv_sd": ..., "test": ..., "ci_low": ..., "ci_high": ...}, ...])

## Part B · Tuning and the choice of validation scheme

### B1 · The ridge penalty for the description length

With about 30 dummy columns ridge regression has little to shrink. We add all pairwise products (`PolynomialFeatures(degree=2)`, for example language × section), which gives the penalty something to control, and search `alpha` on a log grid.

In [ ]:
ridge = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler(), Ridge())
grid = GridSearchCV(ridge, {"ridge__alpha": np.logspace(-1, 5, 7)}, cv=cv5, scoring="r2", return_train_score=True)
grid.fit(X_reg[past], y_reg[past])
res = pd.DataFrame(grid.cv_results_)[["param_ridge__alpha", "mean_train_score", "mean_test_score", "std_test_score"]]
print(res.round(4).to_string(index=False))
print("best:", grid.best_params_, " out-of-time R²:", round(r2_score(y_reg[~past], grid.predict(X_reg[~past])), 3))

### B2 · The regularisation `alpha` of the heading classifier: random versus time-based validation

The leaderboard trains on the past and tests on the future. Does a random `KFold` give the same picture as a `TimeSeriesSplit`? We tune `alpha` with both on the development years and compare the chosen value, the CV score and the out-of-time score.

In [ ]:
param = {"sgdclassifier__alpha": [1e-6, 1e-5, 1e-4]}
dev_X, dev_y = dec["description"][past], dec["heading"][past]
schemes = {"random (KFold)": KFold(3, shuffle=True, random_state=0),
           "time (TimeSeriesSplit)": TimeSeriesSplit(3)}
rows = []
for name, splitter in schemes.items():
    gs = GridSearchCV(text_model, param, cv=splitter, scoring="accuracy").fit(dev_X, dev_y)
    rows.append({"scheme": name, "best_alpha": gs.best_params_["sgdclassifier__alpha"],
                 "cv_accuracy": round(gs.best_score_, 3),
                 "all cv scores": gs.cv_results_["mean_test_score"].round(3),
                 "out_of_time_accuracy": round(accuracy_score(dec["heading"][~past], gs.predict(dec["description"][~past])), 3)})
pd.DataFrame(rows)

In [ ]:
# a TF-IDF setting: word unigrams versus unigrams and bigrams, judged by time
for ngrams in [(1, 1), (1, 2)]:
    m = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=ngrams),
                      SGDClassifier(alpha=1e-5, random_state=0, n_jobs=-1)).fit(dev_X, dev_y)
    print(ngrams, "out-of-time accuracy", round(accuracy_score(dec["heading"][~past], m.predict(dec["description"][~past])), 3))

In [ ]:
# per-fold scores of the time-based split: does performance change over the years?
for tr, va in TimeSeriesSplit(5).split(dec):
    text_model.fit(dec["description"].iloc[tr], dec["heading"].iloc[tr])
    s = accuracy_score(dec["heading"].iloc[va], text_model.predict(dec["description"].iloc[va]))
    print(dec["start_date"].iloc[va].min().date(), "to", dec["start_date"].iloc[va].max().date(),
          " training rows", len(tr), " accuracy", round(s, 3))

**Task B.** (1) Which `alpha` was chosen for ridge and for the classifier? Is the optimum flat or sharp? If the best value lies at the edge of the grid, extend the grid and search again. (2) Do random and time-based validation agree on the chosen `alpha`? On the score? Why is the time-based CV score lower than the out-of-time score, although both look into the future (hint: training-set size in the first fold)? (3) In the per-fold scores, two things change at once: the amount of training data and the period. How could you separate them? (4) Which scheme would you use for the leaderboard and why?

### B3 · Renewals: grouped cross-validation

Renewed decisions repeat a description word for word. If one copy is in the training folds and one in the validation fold, the classifier only has to recognise the text. `GroupKFold` with the normalised description as group keeps copies together.

In [ ]:
norm = dec["description"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print("share of descriptions that occur more than once in the sample:", round(norm.duplicated(keep=False).mean(), 3))
random_cv = cross_val_score(text_model, dec["description"], dec["heading"], cv=KFold(3, shuffle=True, random_state=0))
grouped_cv = cross_val_score(text_model, dec["description"], dec["heading"], cv=GroupKFold(3), groups=norm)
print("random 3-fold :", random_cv.round(3), random_cv.mean().round(3))
print("grouped 3-fold:", grouped_cv.round(3), grouped_cv.mean().round(3))

In our run the grouped score was not lower than the random one (0.811 against 0.802): the sample contains few renewal pairs (2.4 % of its descriptions occur more than once), and `GroupKFold` builds different folds, so the difference is noise of the fold assignment. On the full training set, with the same model, a grouped split scored 0.7 points lower than a random one.

**Task.** Is this what you expected? On the full training set 4.6 % of the descriptions repeat; theory page 01 reports the result there. Which matters more for the leaderboard: renewals or time?

## Part C · A leaking workflow and its fix

### C1 · Preprocessing leakage: word selection outside the cross-validation

On 1,000 decisions of the 20 most frequent headings we count words and keep the 100 words most associated with the heading (chi-squared test). The **leaking** version selects the words on all 1,000 decisions and then cross-validates; the **correct** version puts vectoriser, selector and model into one pipeline.

In [ ]:
top20 = dec["heading"].value_counts().index[:20]
pool = dec[dec["heading"].isin(top20)]
small = pool.sample(1000, random_state=0)
clf = LogisticRegression(max_iter=2000)
skf = StratifiedKFold(5, shuffle=True, random_state=0)

# leaking: vocabulary and word selection see all rows, including the later validation folds
counts = CountVectorizer(min_df=2).fit_transform(small["description"])
X_leak = SelectKBest(chi2, k=100).fit_transform(counts, small["heading"])
leak = cross_val_score(clf, X_leak, small["heading"], cv=skf, scoring="accuracy")

# correct: every step is refitted on the training folds only
pipe = make_pipeline(CountVectorizer(min_df=2), SelectKBest(chi2, k=100), clf)
clean = cross_val_score(pipe, small["description"], small["heading"], cv=skf, scoring="accuracy")
print("leaking :", leak.round(3), leak.mean().round(3))
print("pipeline:", clean.round(3), clean.mean().round(3))

Which score is closer to the truth? Check on decisions that neither workflow has seen: 5,000 other decisions of the same 20 headings.

In [ ]:
fresh = pool.drop(small.index).sample(5000, random_state=1)
pipe.fit(small["description"], small["heading"])
print("accuracy on 5,000 fresh decisions:", round(accuracy_score(fresh["heading"], pipe.predict(fresh["description"])), 3))

On these data the gap is small: the selected words carry real signal. With fewer rows, more candidate words or noisier labels it grows (theory page 03 shows an extreme case on pure noise).

### C2 · Target leakage: the customs' justification

`classification_justification` is written by customs when they decide; it names the heading in about 70 % of the decisions. A new request does not have it, and the test set does not contain it. The same is true of `keywords` and `cn_code`.

In [ ]:
print("justification contains the heading:",
      round(np.mean([h in str(j).replace(" ", "") for h, j in zip(dec["heading"], dec["classification_justification"])]), 3))
print(dec["classification_justification"].iloc[0][:300])

tr, va = train_test_split(np.flatnonzero(past), test_size=0.2, random_state=0)
new = np.flatnonzero(~past)
variants = {"description": dec["description"],
            "description + keywords": dec["description"] + " " + dec["keywords"].fillna(""),
            "description + justification": dec["description"] + " " + dec["classification_justification"].fillna("")}
rows = []
for name, X in variants.items():
    text_model.fit(X.iloc[tr], dec["heading"].iloc[tr])
    rows.append({"features": name,
                 "validation accuracy (leak present)": accuracy_score(dec["heading"].iloc[va], text_model.predict(X.iloc[va])),
                 "accuracy in use (description only)": accuracy_score(dec["heading"].iloc[new], text_model.predict(dec["description"].iloc[new]))})
leak_table = pd.DataFrame(rows).round(3)
leak_table

The leaking models look excellent in validation, because the leak is present in every validation row. Applied where they will be used, to descriptions only, they are no better, or worse, than the honest model. Session 9 returns to features that are not available at prediction time.

### C3 · Summary table

In [ ]:
summary = pd.DataFrame([
    {"workflow": "word selection outside CV (leaking)", "cv_accuracy": leak.mean()},
    {"workflow": "word selection inside a Pipeline", "cv_accuracy": clean.mean()},
])
summary.round(3)

**Task C.** (1) Add the rows of C2 to the summary table. (2) Explain in two sentences why the leaking workflows score higher in validation, and which number you would report. (3) Look at the columns of `train.parquet` and `test.parquet`. Which other column of the training data would leak information about the heading, and which columns can be used honestly?

**Team project.** Write a half-page validation plan for your project model: which splitter (and why), which metric, how you will report uncertainty, and how you will avoid leakage.